# Sentinel-2 Snapshots on July 1 – Espírito Santo

Extract yearly Sentinel-2 (L2A/L1C) scenes intersecting Espírito Santo, Brazil, constrained to July 1st for every year from 2015 through 2025. The notebook mirrors the MVP data workflow and stores the assets in `../data/sentinel-2/espirito-santo-jul1/<year>/`.

> ⚠️ This file only defines the workflow; run the import and download cells manually when network access is available.

## Environment preparation
- Python 3.10+
- `requests`
- 1–2 GB free storage for the July-1 stack (≈0.4 GB per scene)

In [1]:
from __future__ import annotations

import datetime as dt
import json
from pathlib import Path
from typing import Dict, Iterable, List, Optional
from urllib.parse import urlparse

import requests

In [2]:
AWS_STAC_URL = "https://earth-search.aws.element84.com/v1"
PREFERRED_COLLECTIONS = ("sentinel-2-l2a", "sentinel-2-l1c")

# Bounding box covering Espírito Santo in EPSG:4326 (lon/lat)
AOI_BOUNDING_BOX = [-41.88, -21.30, -39.52, -17.88]
TARGET_MONTH = 7
TARGET_DAY = 1
JULY_BUFFER_SEQUENCE = (0, 3, 10, 30, 60, None)
MAX_CLOUD_COVER = 20
SEARCH_LIMIT = 100
ASSETS_TO_DOWNLOAD = ["visual", "metadata", "B04", "B08", "SCL"]
CHUNK_SIZE_MB = 8

DATA_ROOT = Path("../data/sentinel-2/espirito-santo-jul1")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_YEARS = list(range(2015, 2026))

SESSION = requests.Session()
SESSION.headers.update(
    {
        "Accept": "application/geo+json",
        "Content-Type": "application/geo+json",
        "User-Agent": "carbono-data-collection/0.1-data-gathering-jul1",
    }
)

## Helper functions
Helpers to build the July-1 windows, query the STAC catalog for the entire year, enforce the Sentinel-2-only constraint, and stream the requested assets to disk. The search widens its tolerance (0 → 3 → 10 → 30 → 60 days) before finally accepting any date in the year if needed, guaranteeing a usable scene.

In [3]:
def format_datetime(value: dt.datetime) -> str:
    return value.strftime("%Y-%m-%dT%H:%M:%SZ")


def target_datetime(year: int) -> dt.datetime:
    return dt.datetime(year, TARGET_MONTH, TARGET_DAY, tzinfo=dt.timezone.utc)


def year_time_range(year: int) -> str:
    start = dt.datetime(year, 1, 1, tzinfo=dt.timezone.utc)
    end = dt.datetime(year, 12, 31, 23, 59, 59, tzinfo=dt.timezone.utc)
    return f"{format_datetime(start)}/{format_datetime(end)}"


def build_search_payload(
    collection: str,
    year: int,
    cloud_threshold: Optional[int],
    limit: int = SEARCH_LIMIT,
) -> Dict:
    payload: Dict[str, object] = {
        "collections": [collection],
        "bbox": AOI_BOUNDING_BOX,
        "datetime": year_time_range(year),
        "limit": limit,
        "sortby": [{"field": "eo:cloud_cover", "direction": "asc"}],
    }
    if cloud_threshold is not None:
        payload["query"] = {"eo:cloud_cover": {"lt": cloud_threshold}}
    return payload


def parse_datetime(value: str | None) -> Optional[dt.datetime]:
    if not value:
        return None
    if value.endswith("Z"):
        value = value.replace("Z", "+00:00")
    try:
        return dt.datetime.fromisoformat(value)
    except ValueError:
        return None


def fetch_year_features(
    year: int,
    collection: str,
    cloud_threshold: Optional[int],
) -> List[Dict]:
    payload = build_search_payload(collection, year, cloud_threshold)
    response = SESSION.post(f"{AWS_STAC_URL}/search", json=payload, timeout=90)
    response.raise_for_status()
    features = response.json().get("features", [])
    return [
        feature
        for feature in features
        if str(feature.get("id", "")).startswith("S2")
    ]


def within_buffer(
    feature: Dict, target_ts: dt.datetime, buffer_days: Optional[int]
) -> bool:
    if buffer_days is None:
        return True
    sensing_time = parse_datetime(feature.get("properties", {}).get("datetime"))
    if not sensing_time:
        return False
    delta_seconds = abs((sensing_time - target_ts).total_seconds())
    return delta_seconds <= buffer_days * 86400


def feature_key(feature: Dict, target_ts: dt.datetime) -> tuple:
    properties = feature.get("properties", {})
    sensing_time = parse_datetime(properties.get("datetime")) or target_ts
    delta = abs((sensing_time - target_ts).total_seconds())
    cloud = properties.get("eo:cloud_cover", 101)
    return (delta, cloud)


def search_best_item(
    year: int,
    collections: Iterable[str] | None = None,
    buffer_sequence: Iterable[Optional[int]] | None = None,
    cloud_sequence: Iterable[Optional[int]] | None = None,
) -> Dict:
    collections = tuple(collections or PREFERRED_COLLECTIONS)
    buffer_sequence = tuple(buffer_sequence or JULY_BUFFER_SEQUENCE)
    cloud_sequence = tuple(cloud_sequence or (MAX_CLOUD_COVER, 60, None))
    target_ts = target_datetime(year)
    last_error: Optional[Exception] = None

    for buffer_days in buffer_sequence:
        for cloud_threshold in cloud_sequence:
            for collection in collections:
                try:
                    features = fetch_year_features(
                        year=year,
                        collection=collection,
                        cloud_threshold=cloud_threshold,
                    )
                except requests.RequestException as exc:
                    last_error = exc
                    continue

                candidates = [
                    feature
                    for feature in features
                    if within_buffer(feature, target_ts, buffer_days)
                ]
                if not candidates:
                    continue

                best = min(candidates, key=lambda f: feature_key(f, target_ts))
                properties = best.setdefault("properties", {})
                sensing_time = parse_datetime(properties.get("datetime"))
                if sensing_time:
                    days_from_target = (sensing_time - target_ts).total_seconds() / 86400
                else:
                    days_from_target = None
                best["collection"] = collection
                best["cloud_threshold"] = cloud_threshold
                best["buffer_days"] = buffer_days
                best["days_from_target"] = days_from_target
                return best

    raise ValueError(last_error or f"No Sentinel-2 scenes found for {year}")


def infer_asset_filename(item_id: str, asset_name: str, asset_href: str) -> str:
    parsed_path = Path(urlparse(asset_href).path)
    suffix = parsed_path.suffix or ".bin"
    clean_item_id = item_id.replace(" ", "_")
    return f"{clean_item_id}_{asset_name}{suffix}"


def download_asset(asset_href: str, destination: Path) -> Path:
    chunk_size = CHUNK_SIZE_MB * 1024 * 1024
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        return destination

    with SESSION.get(asset_href, stream=True, timeout=120) as response:
        response.raise_for_status()
        with destination.open("wb") as target_file:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    target_file.write(chunk)
    return destination


def download_year_assets(
    year: int,
    item: Dict,
    assets: Iterable[str] | None = None,
) -> List[Path]:
    assets = tuple(assets or ASSETS_TO_DOWNLOAD)
    downloaded_paths: List[Path] = []
    year_dir = DATA_ROOT / str(year)

    for asset_name in assets:
        asset = item.get("assets", {}).get(asset_name)
        if not asset:
            print(f"[{year}] Asset '{asset_name}' missing; skipping")
            continue

        asset_href = asset["href"]
        filename = infer_asset_filename(item["id"], asset_name, asset_href)
        local_path = year_dir / filename
        download_asset(asset_href, local_path)
        downloaded_paths.append(local_path)
    return downloaded_paths


def summarize_item(year: int, item: Dict, downloaded_paths: Iterable[Path]) -> Dict:
    properties = item.get("properties", {})
    return {
        "year": year,
        "product_id": item.get("id"),
        "collection": item.get("collection"),
        "mgrs_tile": properties.get("mgrs:tile"),
        "sensing_time": properties.get("datetime"),
        "cloud_cover": properties.get("eo:cloud_cover"),
        "data_coverage": properties.get("sentinel:data_coverage"),
        "cloud_threshold_applied": item.get("cloud_threshold"),
        "buffer_days_used": item.get("buffer_days"),
        "days_from_target": item.get("days_from_target"),
        "downloaded_assets": [str(path) for path in downloaded_paths],
    }

## 1. Query July-1 scenes per year
The next cell scans 2015–2025 and prints the chosen product, collection, sensing timestamp, and cloud coverage. It shows the tolerance used (`buffer_days`) and the actual offset (days from July 1).

In [4]:
yearly_items: Dict[int, Dict] = {}

for year in TARGET_YEARS:
    try:
        item = search_best_item(year)
    except ValueError as exc:
        print(f"[{year}] No usable scene: {exc}")
        continue

    properties = item.get("properties", {})
    buffer_days = item.get("buffer_days")
    delta_days = item.get("days_from_target")
    print(
        "[{year}] {product} | {collection} | cloud={cloud:.1f}% | buffer={buffer} | Δdays={delta:.1f} | {datetime}".format(
            year=year,
            product=item.get("id"),
            collection=item.get("collection"),
            cloud=properties.get("eo:cloud_cover", float("nan")),
            buffer=buffer_days,
            delta=delta_days if delta_days is not None else float("nan"),
            datetime=properties.get("datetime"),
        )
    )
    yearly_items[year] = item

print(f"Collected {len(yearly_items)} scene references out of {len(TARGET_YEARS)} years")

[2015] No usable scene: No Sentinel-2 scenes found for 2015
[2016] No usable scene: No Sentinel-2 scenes found for 2016
[2017] No usable scene: No Sentinel-2 scenes found for 2017
[2018] No usable scene: No Sentinel-2 scenes found for 2018
[2019] No usable scene: No Sentinel-2 scenes found for 2019
[2020] No usable scene: No Sentinel-2 scenes found for 2020
[2021] No usable scene: No Sentinel-2 scenes found for 2021
[2022] No usable scene: No Sentinel-2 scenes found for 2022
[2023] No usable scene: No Sentinel-2 scenes found for 2023
[2024] No usable scene: No Sentinel-2 scenes found for 2024
[2025] No usable scene: No Sentinel-2 scenes found for 2025
Collected 0 scene references out of 11 years


## 2. Download assets + write index
Download the requested bands plus metadata, organized per year, and record the selection metadata under `../data/sentinel-2/espirito-santo-jul1/sentinel2_es_jul1_index.json`.

In [ ]:
index_entries: List[Dict] = []

for year, item in yearly_items.items():
    downloaded_paths = download_year_assets(year, item)
    index_entries.append(summarize_item(year, item, downloaded_paths))
    print(f"[{year}] Saved {len(downloaded_paths)} assets -> {DATA_ROOT / str(year)}")

index_path = DATA_ROOT / "sentinel2_es_jul1_index.json"
with index_path.open("w", encoding="utf-8") as fp:
    json.dump(index_entries, fp, indent=2)

print(f"Metadata index written to {index_path}")

## Next steps
- Validate the per-year sensing time versus the July-1 target; adjust `JULY_BUFFER_SEQUENCE` if you want stricter or looser tolerances.
- Feed the indexed assets into the forest-change analysis pipeline or classification experiments focused on mid-season imagery.
- Extend the asset list with additional spectral bands as modeling needs evolve.